# 01 · Data collection

**Goal:** build every dataset the project needs, up to one fixed *as-of date*, so the analysis can be reproduced exactly.

**Outputs** (in `data/processed/`): `prices.csv`, `iren_volume.csv`, `risk_free.csv`, `fama_french_daily.csv`, `fundamentals_quarterly.csv`. Raw downloads go to `data/raw/` and are *not* committed: this notebook can always rebuild them.

> **Finance note: why an as-of date?** Market data changes every day. If you re-run in a month without a fixed date, every number in your report moves. Professional research always states its *pricing date*.

In [ ]:
import pandas as pd
from iren import config, data

pd.set_option("display.float_format", "{:,.4f}".format)
print("As-of date:", config.AS_OF, "| tickers:", len(config.ALL_TICKERS))

## 1. Prices (Yahoo Finance)

Adjusted close prices account for splits and dividends. IREN pays no dividend, but peers or ETFs might.

In [ ]:
close, volume = data.download_prices(config.ALL_TICKERS)
close.to_csv(config.DATA_RAW / "prices_close_raw.csv")
volume.to_csv(config.DATA_RAW / "volume_raw.csv")
close.tail()

### Quality check

Peers listed at different times (e.g. CoreWeave `CRWV` only in March 2025), so their histories are shorter. That is expected. Missing values *after* a ticker's first date are the ones to investigate.

In [ ]:
check = pd.DataFrame({
    "first_date": close.apply(lambda s: s.first_valid_index()),
    "last_date": close.apply(lambda s: s.last_valid_index()),
    "n_obs": close.count(),
    "missing_after_start": close.apply(lambda s: s.loc[s.first_valid_index():].isna().sum()),
})
check.sort_values("first_date")

### Align to IREN's trading calendar

Bitcoin trades every day and stocks don't. Keeping only IREN's trading days means a Monday BTC return covers Friday→Monday, the same window as IREN's Monday return.

In [ ]:
prices = data.align_to_trading_days(close, anchor=config.TARGET)
prices.to_csv(config.DATA_PROCESSED / "prices.csv")
volume.reindex(prices.index)[[config.TARGET]].to_csv(config.DATA_PROCESSED / "iren_volume.csv")
print(prices.shape, prices.index.min().date(), "->", prices.index.max().date())

## 2. Risk-free rate (FRED)

The 3-month T-bill is the risk-free rate for Sharpe ratios and excess returns. The 10-year Treasury yield is the long-run risk-free rate for the cost of equity in the DCF (notebook 08).

In [ ]:
rf = data.fetch_fred().reindex(prices.index).ffill()
rf["rf_daily"] = rf["tbill3m"] / config.TRADING_DAYS
rf.to_csv(config.DATA_PROCESSED / "risk_free.csv")
rf.tail()

## 3. Fama–French factors

Market, size (SMB), value (HML), profitability (RMW), investment (CMA) and momentum. In notebook 04 these test whether IREN earned **alpha**: return that standard risk factors cannot explain. The library lags by 1-2 months, so the most recent weeks may be missing.

In [ ]:
ff = data.fetch_fama_french()
ff.to_csv(config.DATA_PROCESSED / "fama_french_daily.csv")
print("Fama-French data ends:", ff.index.max().date())
ff.tail()

## 4. Financial statements (SEC EDGAR XBRL)

Every number in a 10-K / 10-Q is tagged in XBRL, and the SEC's free API returns all of them. **Before running:** put your name and email in `SEC_USER_AGENT` in `src/iren/config.py` (the SEC requires it).

> **Trap:** IREN used to file as a foreign private issuer (20-F / 6-K, IFRS tags) and later switched to US GAAP 10-K/10-Q. Expect concept names to change and older quarters to be missing. Fill gaps from the press releases in `data/manual/kpis.csv`.

In [ ]:
facts = data.fetch_companyfacts()
long = data.companyfacts_to_long(facts)
long.to_csv(config.DATA_RAW / "edgar_companyfacts_long.csv", index=False)
long["taxonomy"].value_counts()

In [ ]:
inventory = (long.groupby(["taxonomy", "concept", "label"])
                 .agg(n=("value", "size"), first=("end", "min"), last=("end", "max"))
                 .reset_index().sort_values("last", ascending=False))

def find(term):
    """Search concept names and labels, e.g. find('revenue')."""
    m = (inventory["concept"].str.contains(term, case=False)
         | inventory["label"].str.contains(term, case=False, na=False))
    return inventory[m]

find("revenue")

### Map the concepts you need

Use `find()` to check the exact names IREN uses and edit the candidate lists below. The first name in each list wins, and later names fill gaps. Cash-flow items are reported year-to-date in 10-Qs; `quarterly_series` converts them to single quarters for you.

In [ ]:
CONCEPTS = {
    # our column        : ([candidate XBRL concepts], kind)
    "revenue":          (["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "Revenue"], "flow"),
    "net_income":       (["NetIncomeLoss", "ProfitLoss"], "flow"),
    "operating_income": (["OperatingIncomeLoss"], "flow"),
    "d_and_a":          (["DepreciationDepletionAndAmortization", "DepreciationAndAmortization"], "flow"),
    "impairment":       (["ImpairmentOfLongLivedAssetsHeldForUse", "AssetImpairmentCharges"], "flow"),
    "interest_expense": (["InterestExpense", "InterestExpenseNonoperating"], "flow"),
    "operating_cf":     (["NetCashProvidedByUsedInOperatingActivities"], "flow"),
    "capex":            (["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets"], "flow"),
    "cash":             (["CashAndCashEquivalentsAtCarryingValue"], "instant"),
    "total_assets":     (["Assets"], "instant"),
    "current_assets":   (["AssetsCurrent"], "instant"),
    "current_liab":     (["LiabilitiesCurrent"], "instant"),
    "total_liab":       (["Liabilities"], "instant"),
    "equity":           (["StockholdersEquity"], "instant"),
    "long_term_debt":   (["LongTermDebtNoncurrent", "ConvertibleNotesPayable", "LongTermDebt"], "instant"),
}

fund = pd.DataFrame({name: data.first_available(long, cands, kind)
                     for name, (cands, kind) in CONCEPTS.items()})
fund = fund.loc["2021-06-30":config.AS_OF]
fund.index.name = "period_end"
fund.insert(0, "fiscal_q", [data.fiscal_quarter_label(d) for d in fund.index])
fund.to_csv(config.DATA_PROCESSED / "fundamentals_quarterly.csv")
fund.tail(8)

### Reconcile with the press releases

If EDGAR and the releases disagree, find out why before moving on (a restatement, a label change, or a YTD figure). Write the explanation in `docs/data_dictionary.md`.

In [ ]:
kpis = pd.read_csv(config.DATA_MANUAL / "kpis.csv", parse_dates=["period_end"]).set_index("period_end")
recon = kpis[["fiscal_q", "total_revenue_usdm", "net_income_usdm"]].join(fund[["revenue", "net_income"]] / 1e6)
recon["revenue_diff_usdm"] = recon["total_revenue_usdm"] - recon["revenue"]
recon

## 5. Hand-collected files

These come from IREN's results releases and 10-K / 10-Q notes. Every row needs a `source_url`. Your job here:

- [ ] `kpis.csv`: extend back to FY24Q1 (revenue by segment, MW, EH/s, ARR, backlog, shares outstanding)
- [ ] `convertible_notes.csv`: fill missing conversion prices from the 10-K debt note; set `verified_against_10k` to `yes`
- [ ] `events.csv`: confirm each announcement date (use the first trading day the market could react)

In [ ]:
notes = pd.read_csv(config.DATA_MANUAL / "convertible_notes.csv")
events = pd.read_csv(config.DATA_MANUAL / "events.csv", parse_dates=["event_date"])
for name, df in {"kpis": kpis, "notes": notes, "events": events}.items():
    missing = df["source_url"].isna().sum() if "source_url" in df else "n/a"
    print(f"{name:6s} rows={len(df):3d}  rows without source={missing}")
notes

## So what for the investor?

*Write 3-5 sentences in plain English. Use numbers from this notebook. Notebook 09 is built from these cells, so make each sentence count.*

- 
- 
- 

## Commit checkpoint

1. **Kernel → Restart → Run All** and check there are no errors.
2. Source Control panel (Ctrl+Shift+G): review the diff, then commit with messages like:
   - `data: download prices for IREN, BTC, market and peers`
   - `data: add EDGAR quarterly fundamentals`
   - `data: hand-collect KPIs FY24Q1-FY27Q1 with sources`
   - `docs: add data dictionary`
3. `git push -u origin nb01-data`, then open a pull request on GitHub and merge it.
